# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data, written exactly as Python's `json.dumps` writes it. These
cases cover strictness on output and input, text forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import { JSON, Plain, Proxies, Schemas as S } from "@mbse/schemas/Framework";
import { DecodeError, ValueError } from "@mbse/schemas/Framework/Errors";
import type { PlainMap } from "@mbse/schemas/Framework/Plain";
import { assert, equal, map, raises, same_graph, text } from "./support.js";

const store = new Proxies.OfStore();

let Item = new S.OfObject.Builder().name("Item").ref().properties(text("s"), text("i", BigInt), text("f", Number), text("b", Boolean),
  text("raw", Uint8Array)).create();
const Link = new S.OfRelation.Builder().name("Link").links("from", "to").properties(text("w", Number)).create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("out").of(Link).me("from"), (r) => r.name("in").of(Link).me("to")).update();
store.register(Item);
store.register(Link);
const B = store;
const FromJSON = JSON.FromJSON(B);
const field = (textOut: string, name: string) =>
  (((JSON.loads(textOut) as PlainMap).get("objects") as PlainMap).get("s0") as PlainMap).get(name);

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for (const value of [NaN, Infinity, -Infinity]) {
  const out = JSON.ToJSON(store)(Item, B.Item().f(value).create());
  assert(!/\b(NaN|Infinity)\b(?!")/.test(out.replaceAll('"NaN"', "").replaceAll('"Infinity"', "").replaceAll('"-Infinity"', "")));
  assert(["NaN", "Infinity", "-Infinity"].includes(field(out, "f") as string));
}
raises(ValueError, () => JSON.dumps(map({ x: NaN }))); // plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for (const bad of ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', "[NaN]",
  '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
  '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', "", "   ", '{"a": 1} {"b": 2}', '{"a": .5}']) {
  raises(DecodeError, () => JSON.loads(bad), "line 1, column ");
}

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
function encode(value: string, encoding: string): Uint8Array {
  const units = [...value].flatMap((c) => {
    const p = c.codePointAt(0) as number;
    return p > 0xffff ? [0xd800 + ((p - 0x10000) >> 10), 0xdc00 + ((p - 0x10000) & 0x3ff)] : [p];
  });
  const points = [...value].map((c) => c.codePointAt(0) as number);
  const u16 = (le: boolean) => units.flatMap((u) => (le ? [u & 0xff, u >> 8] : [u >> 8, u & 0xff]));
  const u32 = (le: boolean) => points.flatMap((p) => { const b = [p >>> 24, (p >> 16) & 0xff, (p >> 8) & 0xff, p & 0xff]; return le ? b.reverse() : b; });
  switch (encoding) {
    case "utf-16": return new Uint8Array([0xff, 0xfe, ...u16(true)]);
    case "utf-16-le": return new Uint8Array(u16(true));
    case "utf-16-be": return new Uint8Array(u16(false));
    case "utf-32": return new Uint8Array([0xff, 0xfe, 0x00, 0x00, ...u32(true)]);
    case "utf-8-sig": return new Uint8Array([0xef, 0xbb, 0xbf, ...new TextEncoder().encode(value)]);
    default: return new TextEncoder().encode(value);
  }
}
{
  const doc = '{"a": "é"}';
  assert(equal(JSON.loads(doc), map({ a: "é" })) && equal(JSON.loads(encode(doc, "utf-8")), map({ a: "é" })));
  for (const encoding of ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]) {
    assert(equal(JSON.loads(encode(doc, encoding)), map({ a: "é" })), encoding);
  }
  raises(DecodeError, () => JSON.loads("﻿" + doc), "line 1, column 1: unexpected byte order mark"); // a BOM inside a str is an error; in bytes it is accepted
  raises(DecodeError, () => JSON.loads(new Uint8Array([0xff, 0xfe, 0x00])), "input is not valid utf-16"); // undecodable bytes
}

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
{
  const item = B.Item().raw(new Uint8Array([1])).s("日本").i(1n).create();
  const compact = JSON.ToJSON(store)(Item, item);
  assert(compact === '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}', compact);
  const pretty = JSON.ToJSON(store)(Item, item, { indent: 2 });
  assert(pretty.includes("\n") && equal(JSON.loads(pretty), JSON.loads(compact)));
  assert(JSON.ToJSON(store).OfObject(Item, item, { indent: 4 }) === JSON.dumps(Plain.ToPlain(store)(Item, item), { indent: 4 }));
}

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for (const value of [0n, -1n, 2n ** 53n + 1n, 2n ** 100n, -(2n ** 200n)]) {
  assert((FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().i(value).create())) as any).i === value);
}
for (const value of [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]) {
  const back = (FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().f(value).create())) as any).f;
  assert(Object.is(back, value));
}
for (const value of ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\u{1f600}", "a\r\nb"]) {
  assert((FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().s(value).create())) as any).s === value, value);
}
{
  const all = new Uint8Array(Array.from({ length: 256 }, (_, i) => i));
  assert(equal((FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().raw(all).create())) as any).raw, all));
}

## JSN-06 · A lone surrogate is valid JSON text in memory but not encodable as UTF-8 (pinned)
Python raises when encoding it; JavaScript's `TextEncoder` silently replaces it with U+FFFD. Either way, callers
writing files should reject such strings.

In [ ]:
{
  const lone = JSON.ToJSON(store)(Item, B.Item().s("\ud800").create());
  assert(lone.includes("\ud800"));
  const encoded = new TextDecoder().decode(new TextEncoder().encode(lone));
  assert(!encoded.includes("\ud800") && encoded.includes("�")); // lossy, where Python raises UnicodeEncodeError
}

## JSN-07 · Natives at the top level

In [ ]:
assert(JSON.ToJSON(store)(new S.OfNative.Data(BigInt), 5n) === "5");
assert(JSON.ToJSON(store)(new S.OfNative.Data(Uint8Array), new Uint8Array([0xff])) === '"/w=="');
assert(JSON.ToJSON(store).OfNative(new S.OfNative.Data(Number), Infinity) === '"Infinity"');
assert(FromJSON(new S.OfNative.Data(String), '"x"') === "x" && FromJSON.OfNative(new S.OfNative.Data(String), '"x"') === "x");
assert(JSON.ToJSON(store)(new S.OfNative.Data(Number), 1.0) === "1.0"); // a float stays visibly a float
raises(DecodeError, () => FromJSON(new S.OfNative.Data(BigInt), "true"), "$: expected int, got bool");
raises(DecodeError, () => FromJSON(new S.OfNative.Data(Number), "1"), "$: expected float, got int"); // JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
{
  const hub = B.Item().s("hub").out((x: any) => x.to((y: any) => y.s("leaf1")).w(1.0))
    .out((x: any) => x.to((y: any) => y.s("leaf2")).w(NaN)).create();
  const graph = Plain.ToPlain(store).Reachable(Item, hub);
  const text_ = JSON.ToJSON(store).Reachable(Item, hub);
  assert(equal(JSON.loads(text_), graph));
  const back = FromJSON.Reachable(Item, text_) as Proxies.Instance;
  assert(back !== hub && same_graph(Plain.ToPlain(store).Reachable(Item, back), graph));
  raises(DecodeError, () => FromJSON(Item, JSON.ToJSON(store)(Item, hub)), "$.objects.s0.out[0].to: unresolved reference 's1'");
  raises(DecodeError, () => FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"')), "unresolved reference 's9'");
}

## JSN-09 · Every input problem is a `DecodeError` with the same reason and position in every binding
The table is shared verbatim with the Python suite. Rows: input (text, or bytes as a list), the expected message
(or null when it parses), and Python's repr of the parsed value. Columns count code points; undecodable bytes have
no location.

In [ ]:
import { repr } from "@mbse/schemas/Framework/Repr";
{
  const TABLE = globalThis.JSON.parse(String.raw`[["", "line 1, column 1: expecting a value", null], ["   ", "line 1, column 4: expecting a value", null], ["-", "line 1, column 1: expecting a value", null], ["nul", "line 1, column 1: expecting a value", null], ["tru", "line 1, column 1: expecting a value", null], [".5", "line 1, column 1: expecting a value", null], ["[1,]", "line 1, column 3: trailing comma before ']'", null], ["{\"a\": 1,}", "line 1, column 8: trailing comma before '}'", null], ["{\"a\" 1}", "line 1, column 6: expecting ':'", null], ["{1: 2}", "line 1, column 2: expecting a property name in double quotes", null], ["{\"a\": 1 \"b\": 2}", "line 1, column 9: expecting ',' or '}'", null], ["[1 2]", "line 1, column 4: expecting ',' or ']'", null], ["\"abc", "line 1, column 1: unterminated string", null], ["\"a\\", "line 1, column 1: unterminated string", null], ["\"a\\x\"", "line 1, column 3: invalid escape", null], ["\"\\u12\"", "line 1, column 3: invalid \\uXXXX escape", null], ["\"\\uZZZZ\"", "line 1, column 3: invalid \\uXXXX escape", null], ["\"a\nb\"", "line 1, column 3: invalid control character in a string", null], ["\"a\tb\"", "line 1, column 3: invalid control character in a string", null], ["1 2", "line 1, column 3: unexpected data after the value", null], ["{\"a\": 1} x", "line 1, column 10: unexpected data after the value", null], ["[", "line 1, column 2: expecting a value", null], ["{", "line 1, column 2: expecting a property name in double quotes", null], ["{\"a\"", "line 1, column 5: expecting ':'", null], ["{\"a\":", "line 1, column 6: expecting a value", null], ["01", "line 1, column 2: unexpected data after the value", null], ["1.", "line 1, column 2: unexpected data after the value", null], ["1e", "line 1, column 2: unexpected data after the value", null], ["[\n  1,\n  x]", "line 3, column 3: expecting a value", null], ["{\n\"a\": [\n1,\n2\n]\n,}", "line 6, column 1: trailing comma before '}'", null], ["NaN", "line 1, column 1: NaN is not valid JSON", null], ["-Infinity", "line 1, column 1: -Infinity is not valid JSON", null], ["[Infinity]", "line 1, column 2: Infinity is not valid JSON", null], ["99999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999", "line 1, column 1: an integer has 4301 digits, more than the limit of 4300", null], [[0, 0, 0, 49], null, "1"], [[49, 0, 0, 0], null, "1"], [[254, 255, 0, 49], null, "1"], [[0, 49], null, "1"], [[49, 0], null, "1"], [[0, 0, 254, 255, 0, 0, 0, 49], null, "1"], [[0, 0, 0], "line 1, column 1: expecting a value", null], [[255, 254, 49], "input is not valid utf-16", null], [[195, 40], "input is not valid utf-8", null], [[255, 254, 0, 0, 0, 0, 17, 0], "input is not valid utf-32", null], [[0, 0, 0, 49, 50], "input is not valid utf-32-be", null], [[0, 0, 0, 49, 50, 51], "input is not valid utf-32-be", null], [[239, 187, 191, 49], null, "1"], ["\"🚀\\x\"", "line 1, column 3: invalid escape", null], ["{\"a\": 1, \"a\": 2}", "line 1, column 10: duplicate key 'a'", null], ["{\"a\": {\"b\": 1, \"b\": 2}}", "line 1, column 16: duplicate key 'b'", null], ["[1, NaN]", "line 1, column 5: NaN is not valid JSON", null], ["﻿1", "line 1, column 1: unexpected byte order mark", null], ["\n\n  [1,]", "line 3, column 5: trailing comma before ']'", null], ["{\"a\": 1}\r\n x", "line 2, column 2: unexpected data after the value", null], ["\"\\ud83d\"", null, "'\\ud83d'"], ["[1] [2]", "line 1, column 5: unexpected data after the value", null], [[239, 187, 191, 91, 93], null, "[]"], [[254, 255, 216, 0], "input is not valid utf-16", null], [[255], "input is not valid utf-8", null], ["[\"🚀🚀\", x]", "line 1, column 8: expecting a value", null], [[0, 0, 216, 0, 0, 0, 0, 49], "input is not valid utf-32-be", null], [[237, 160, 128], "input is not valid utf-8", null]]`) as [string | number[], string | null, string | null][];
  assert(TABLE.length === 62);
  for (const [source, message, result] of TABLE) {
    const data = Array.isArray(source) ? new Uint8Array(source) : source;
    let value: unknown;
    try {
      value = JSON.loads(data);
    } catch (error) {
      if (!(error instanceof DecodeError)) throw error;
      assert(message !== null, `${String(source)} -> ${error.message}`);
      assert(error.message === message && !error.message.includes("\n"), `${String(source)}: ${error.message}`);
      assert((error.line === null) === message.startsWith("input is not valid") && error.path === null);
      continue;
    }
    assert(message === null && repr(value) === result, `${String(source)} parsed as ${repr(value)}`);
  }
  assert(equal(JSON.loads(String.raw`["\ud83d\ude00", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t\/\\\""]`),
    ["\u{1f600}", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t/\\\""]));
}

## JSN-10 · Only plain data can be written
Python's `json` would turn an int, float, bool or None key into a string; plain data only has string keys.

In [ ]:
raises(TypeError, () => JSON.dumps(new Map([[1n, 2n]]) as never), "keys must be str, not int");
raises(TypeError, () => JSON.dumps(map({ a: [{ b: new Map([[1n, 2n]]) }] })), "keys must be str, not int");
for (const [value, name] of [[new Set([1n, 2n]), "set"], [new Uint8Array([120]), "bytes"], [{}, "object"]] as const) {
  raises(TypeError, () => JSON.dumps(map({ a: [] }).set("a", [value] as never)), `Object of type ${name} is not JSON serializable`);
}

## JSN-11 · Every kind of plain value is written and read, in Python's exact text

In [ ]:
{
  const value = [null, true, false, 0n, -1n, 1.5, "s", [], new Map(), map({ a: [null, {}] }), [[]]];
  assert(JSON.dumps(value) === '[null, true, false, 0, -1, 1.5, "s", [], {}, {"a": [null, {}]}, [[]]]');
  assert(JSON.dumps(value, { indent: 1 }) === '[\n null,\n true,\n false,\n 0,\n -1,\n 1.5,\n "s",\n [],\n {},\n {\n  "a": [\n   null,\n   {}\n  ]\n },\n [\n  []\n ]\n]');
  assert(equal(JSON.loads(JSON.dumps(value)), value) && equal(JSON.loads(JSON.dumps(value, { indent: 1 })), value));
  assert(JSON.loads(" null ") === null && equal(JSON.loads("{}"), new Map()) && equal(JSON.loads("[ ]"), []) && equal(JSON.loads("{ }"), new Map()));
}